In [1]:
# to build a vanilla GAN to generate fake faces

In [2]:
!pip install torchvision

In [4]:
import torch
import os
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image # python imaging library

In [13]:
# images load => transform(normalize,crop,tensor-conversion) => dataset of all images

class ImageProcessor:
    def __init__(self,root_dir_path, transformations = None):
        self.root_dir_path = root_dir_path
        self.transformations = transformations

        # list of paths of all images
        self.all_img_paths = [os.path.join(root_dir_path, img) for img in os.listdir(root_dir_path)]

    # data loader expects following two functions for processing
    
    def __len__(self): #returns length of dataset
        return len(self.all_img_paths)

    def __getitem__(self,idx): # idx (index) is handeled by the dataloader itself
        img_path = self.all_img_paths[idx]
        img = Image.open(img_path).convert("RGB")

        if self.transformations:
            img = self.transformations(img)

        return img
        
        

In [14]:
root_dir_path = "./img_align_celeba"


transformations = transforms.Compose([
    transforms.CenterCrop(178), # 178x218 => 178x178
    transforms.Resize(64), # 64x64
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5)) # [-1,+1]
])

In [16]:
dataset = ImageProcessor(root_dir_path, transformations)

print(len(dataset))


99231


In [17]:
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

## Generator Network

In [18]:
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [25]:
class Generator(nn.Module):
    def __init__(self, z_dim = 100, img_channels = 3):
        super(Generator, self).__init__()

        #fully connected layer

        self.model = nn.Sequential(
            nn.Linear(z_dim, 256), # 100=>256
            nn.ReLU(),

            nn.Linear(256, 512),
            nn.ReLU(),

            nn.Linear(512, 1024),
            nn.ReLU(),

            nn.Linear(1024, 64*64* img_channels),
            nn.Tanh() # tanh returns values between [-1, +1]
        )

    def forward(self,z):
        img = self.model(z)
        img = img.view(img.size(0), 3, 64,64) # img.size(0) gives batch size
        # we are generating images of dim 3*64*64 batch wise

        # fake img -> 64*64*3*128(batchsize) - total tensor (4d)

        return img

## Discriminator Network

In [26]:
class Discriminator(nn.Module):
    def __init__(self, img_channels = 3):
        super(Discriminator, self).__init__()

        #fully connected layer

        self.model = nn.Sequential(
            nn.Flatten(),
            
            nn.Linear(img_channels*64*64, 1024),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Linear(256,1),
            nn.Sigmoid() # tanh returns values between [-1, +1]
        )

    def forward(self,img):
        return self.model(img)

In [27]:
GAN_loss = nn.BCELoss()

generator = Generator()
g_optim = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))

discriminator = Discriminator()
d_optim = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

In [29]:
import torch

if torch.cuda.is_available():
    device= torch.device("cuda")
else:
    device= torch.device("cpu")

print(device)

cpu


In [30]:
generator = generator.to(device)
discriminator = discriminator.to(device)

## Training

In [50]:
def train(generator, discriminator, dataloader, epochs=2):
    for epoch in range(epochs):
        for i, imgs in enumerate(dataloader):
            real_imgs = imgs.to(device)
            batch_size = real_imgs.size(0)

            # creates real images labels & fake labels
            real_labels = torch.ones(batch_size,1).to(device) #[1,1,1,1,....]
            fake_labels = torch.zeros(batch_size,1).to(device) #[0,0,0,0,0,0,....]


            #train the discriminator first

            d_optim.zero_grad()


            fake_imgs = generator(torch.randn(batch_size, 100).to(device))

            real_loss = GAN_loss(discriminator(real_imgs), real_labels)
            fake_loss = GAN_loss(discriminator(fake_imgs.detach()), fake_labels)

            d_loss = (real_loss + fake_loss)/2

            d_loss.backward()
            d_optim.step()


            # train the generator

            g_optim.zero_grad()

            g_loss = GAN_loss(discriminator(fake_imgs), real_labels)

            g_loss.backward()
            g_optim.step()

            if i%50 == 0:
                print(f"for epoch : {epoch+1}/{epochs}.... batch: {i+1}....G-loss:{g_loss}......D-loss: {d_loss} ")
        #save generated imgs for each epoch

        save_generated_img(generator, epoch, device)

In [53]:
import matplotlib.pyplot as plt
import torchvision

def save_generated_img(generator, epoch, device, num_img=8):
    z = torch.randn(num_img, 100).to(device)
    generated_imgs = generator(z).detach().cpu()

    grid = torchvision.utils.make_grid(generated_imgs, nrow, normalize = True)

    plt.imshow(np.transpose(grid, (1, 2, 0)))
    plt.axis("off")
    plt.show()

In [55]:
#train(generator, discriminator, dataloader, epochs=1)